# 03 - Train/Test Split

Stratified 80/20 split of the cleaned data, done **before** WoE/IV so that bins and WoE values are fitted on train only and then applied unchanged to test (no data leakage).

Stratifying on `TARGET` keeps the ~8% default rate identical in both sets.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add project root to sys.path so `src` can be imported from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.split import split_summary, stratified_split

DATA_DIR = PROJECT_ROOT / "data"

## 2. Load cleaned data

In [2]:
# Output of 02_data_cleaning
df = pd.read_pickle(DATA_DIR / "application_train_clean.pkl")
df.shape

(307511, 123)

## 3. Split

In [3]:
# Fixed seed -> the same split every run
train, test = stratified_split(df, target_col="TARGET", test_size=0.2, random_state=42)
train.shape, test.shape

((246008, 123), (61503, 123))

## 4. Verify

In [4]:
# Size and default rate per split: default_rate should match across rows
split_summary({"full": df, "train": train, "test": test}, total=len(df)).round(4)

,n_rows,share,n_default,default_rate
full,307511,1.0,24825,0.0807
train,246008,0.8,19860,0.0807
test,61503,0.2,4965,0.0807


In [5]:
# No client appears in both sets, and no row is lost
assert set(train["SK_ID_CURR"]).isdisjoint(test["SK_ID_CURR"])
assert len(train) + len(test) == len(df)
print("OK: disjoint and complete")

OK: disjoint and complete


## 5. Save

In [6]:
train.to_pickle(DATA_DIR / "train.pkl")
test.to_pickle(DATA_DIR / "test.pkl")
print("Saved train.pkl and test.pkl to", DATA_DIR)

Saved train.pkl and test.pkl to /home/mdtt/Documents/Mdtt/IT/Projects/credit-risk-scorecard/data
